# Collecte des données de Bourse

## Librairies et Fonctions

### Librairies

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import datetime
from types import SimpleNamespace

### Fonctions de chargement des données financières

In [2]:
def charge_donnees_bourse(ticker):
    action_suivie = yf.Ticker(ticker)
    # Intraday : 1 jour si dispo, sinon 2 jours
    day_vue1 = action_suivie.history(period="1d", interval="1m")
    day_vue2 = action_suivie.history(period="2d", interval="1m")
    day_vue = day_vue1 if len(day_vue1) > 0 else day_vue2
    day_vue = day_vue.sort_index()
    # Historique 2 ans
    historique = action_suivie.history(period="2y", interval="1d").sort_index()
    # Indicateurs historique
    historique["prev_close"] = historique["Close"].shift(1)
    historique["MA20"] = historique["Close"].rolling(20).mean()
    historique["MA50"] = historique["Close"].rolling(50).mean()
    historique["support"] = historique["Low"].rolling(50).min()
    historique["resistance"] = historique["High"].rolling(50).max()
    # Vue 1 an
    year_vue = historique.loc[historique.index >= historique.index.max() - pd.Timedelta(days=365)]
    # Indicateurs rendement
    rendement = pd.DataFrame(index=[0])
    risque = pd.DataFrame(index=[0])
    rendement['MA50'] = year_vue['MA50'].iloc[-1]    
    rendement['Delta MA50'] = (year_vue['Close'].iloc[-1] / year_vue['MA50'].iloc[-1] - 1 if year_vue['MA50'].iloc[-1] != 0 else None)
    rendement['ATH / 1 an'] = year_vue['Close'].max()
    rendement['Delta ATH'] = (year_vue['Close'].iloc[-1] / year_vue['Close'].max() - 1 if year_vue['Close'].max() != 0 else None)
    rendement['Cours moyen / 1 an'] = year_vue['Close'].mean()
    rendement['Delta cours moyen'] = (year_vue['Close'].iloc[-1] / year_vue['Close'].mean() - 1 if year_vue['Close'].max() != 0 else None)
    rendement['Dividendes'] = year_vue['Dividends'].sum()
    rendement['Div %'] = rendement.apply(lambda row: row['Dividendes'] / row['Cours moyen / 1 an'] if row['Cours moyen / 1 an'] != 0 else None, axis=1)
    rendement['Rendement'] = (year_vue['Close'].iloc[-1] / year_vue['Close'].iloc[0] - 1 if year_vue['Close'].iloc[0] != 0 else None)
    rendements_jours = year_vue['Close'].pct_change().dropna()
    rendement['Volatilité'] = rendements_jours.std() * (252 ** 0.5)
    rendement['Evol CA 0y'] = action_suivie.revenue_estimate.loc["0y", "growth"]
    rendement['Evol CA +1y'] = action_suivie.revenue_estimate.loc["+1y", "growth"]
    rendement['Evol EPS 0y'] = action_suivie.earnings_estimate.loc["0y", "growth"]
    rendement['Evol EPS +1y'] = action_suivie.earnings_estimate.loc["+1y", "growth"]
    rendement['Potentiel'] = action_suivie.analyst_price_targets['mean'] / action_suivie.analyst_price_targets['current'] -1
    return SimpleNamespace(action_suivie = action_suivie, day_vue = day_vue, year_vue = year_vue, rendement = rendement, risque = risque)

### Fonctions de normalisation des indicateurs

In [3]:
def score(val, obj_min, obj_max, cat_min, cat_max):
    #Retourne un score entre 0 et 1 représentant la distance entre un indicateur et sa zone objectif (0) ou catastrophique (1).
    # Vérification de cohérence des bornes
    if not (cat_min <= obj_min <= obj_max <= cat_max):
        raise ValueError(f"Incohérence dans les bornes reçues : cat_min={cat_min}, obj_min={obj_min}, obj_max={obj_max}, cat_max={cat_max}")
    # 1. Zone idéale → score = 0
    if obj_min <= val <= obj_max:
        return 0.0
    # 2. Zone catastrophique → score = 1
    if val <= cat_min or val >= cat_max:
        return 1.0
    # 3. Zone intermédiaire → interpolation linéaire
    if val < obj_min: # Cas inférieur à l'objectif
        return (obj_min - val) / (obj_min - cat_min)
    if val > obj_max: # Cas supérieur à l'objectif
        return (val - obj_max) / (cat_max - obj_max)
    # Sécurité (ne devrait jamais arriver)
    return None

def safe_get_value(df, col): # Sécurisation de la présence d'un indicateur
    try:
        return df.iloc[0][col]
    except KeyError:
        return None

## Collecte des données

### Choix du titre

In [4]:
# Définir le ticker pour LVMH sur Euronext Paris
ticker = "BNP.PA"

### Récupérations des données financières du titre

In [5]:
# Récupérer les données de l'action
action_suivie = charge_donnees_bourse(ticker)
action_suivie.rendement.keys()

Index(['MA50', 'Delta MA50', 'ATH / 1 an', 'Delta ATH', 'Cours moyen / 1 an',
       'Delta cours moyen', 'Dividendes', 'Div %', 'Rendement', 'Volatilité',
       'Evol CA 0y', 'Evol CA +1y', 'Evol EPS 0y', 'Evol EPS +1y',
       'Potentiel'],
      dtype='object')

## Eléments de décisions pour une recommandation

**Catégorie Dynamique**
Évol CA 0y | Évol CA +1y | Évol EPS 0y | Évol EPS +1y | Delta MA50 | Delta ATH | Delta cours moyen

**Catégorie Gains**
Div % | Rendement | Potentiel

**Catégorie Risques**
Volatilité | Delta MA50 | Delta ATH | Delta cours moyen

### Définition des poids des indicateurs et des catégories

In [6]:
data_poids = [
    # --- Profil PRUDENT ---
    ("prudent", "dynamique", "Delta MA50",        0.15, 0.20),
    ("prudent", "dynamique", "Delta ATH",         0.10, 0.20),
    ("prudent", "dynamique", "Delta cours moyen", 0.20, 0.20),
    ("prudent", "dynamique", "Evol CA 0y",        0.25, 0.20),
    ("prudent", "dynamique", "Evol CA +1y",       0.03, 0.20),
    ("prudent", "dynamique", "Evol EPS 0y",       0.25, 0.20),
    ("prudent", "dynamique", "Evol EPS +1y",      0.02, 0.20),
    ("prudent", "gains", "Div %",     0.50, 0.30),
    ("prudent", "gains", "Rendement", 0.30, 0.30),
    ("prudent", "gains", "Potentiel", 0.20, 0.30),
    ("prudent", "risques", "Volatilité",        0.50, 0.50),
    ("prudent", "risques", "Delta cours moyen", 0.25, 0.50),
    ("prudent", "risques", "Delta MA50",        0.15, 0.50),
    ("prudent", "risques", "Delta ATH",         0.10, 0.50),

    # --- Profil ÉQUILIBRÉ ---
    ("équilibré", "dynamique", "Delta MA50",        0.15, 0.40),
    ("équilibré", "dynamique", "Delta ATH",         0.10, 0.40),
    ("équilibré", "dynamique", "Delta cours moyen", 0.20, 0.40),
    ("équilibré", "dynamique", "Evol CA 0y",        0.20, 0.40),
    ("équilibré", "dynamique", "Evol CA +1y",       0.10, 0.40),
    ("équilibré", "dynamique", "Evol EPS 0y",       0.20, 0.40),
    ("équilibré", "dynamique", "Evol EPS +1y",      0.05, 0.40),
    ("équilibré", "gains", "Div %",     0.35, 0.35),
    ("équilibré", "gains", "Rendement", 0.35, 0.35),
    ("équilibré", "gains", "Potentiel", 0.30, 0.35),
    ("équilibré", "risques", "Volatilité",        0.40, 0.25),
    ("équilibré", "risques", "Delta cours moyen", 0.30, 0.25),
    ("équilibré", "risques", "Delta MA50",        0.20, 0.25),
    ("équilibré", "risques", "Delta ATH",         0.10, 0.25),
    # --- Profil DYNAMIQUE ---
    ("dynamique", "dynamique", "Delta MA50",        0.20, 0.50),
    ("dynamique", "dynamique", "Delta ATH",         0.15, 0.50),
    ("dynamique", "dynamique", "Delta cours moyen", 0.10, 0.50),
    ("dynamique", "dynamique", "Evol CA 0y",        0.03, 0.50),
    ("dynamique", "dynamique", "Evol CA +1y",       0.25, 0.50),
    ("dynamique", "dynamique", "Evol EPS 0y",       0.02, 0.50),
    ("dynamique", "dynamique", "Evol EPS +1y",      0.25, 0.50),
    ("dynamique", "gains", "Div %",     0.20, 0.35),
    ("dynamique", "gains", "Rendement", 0.30, 0.35),
    ("dynamique", "gains", "Potentiel", 0.50, 0.35),
    ("dynamique", "risques", "Volatilité",        0.30, 0.15),
    ("dynamique", "risques", "Delta cours moyen", 0.15, 0.15),
    ("dynamique", "risques", "Delta MA50",        0.25, 0.15),
    ("dynamique", "risques", "Delta ATH",         0.30, 0.15),
]

df_poids = pd.DataFrame(data_poids, columns=["profil", "categorie", "indicateur", "poids_indicateur", "poids_categorie"])
df_poids

,profil,categorie,indicateur,poids_indicateur,poids_categorie
0,prudent,dynamique,Delta MA50,0.15,0.20
1,prudent,dynamique,Delta ATH,0.10,0.20
2,prudent,dynamique,Delta cours moyen,0.20,0.20
3,prudent,dynamique,Evol CA 0y,0.25,0.20
4,prudent,dynamique,Evol CA +1y,0.03,0.20
5,prudent,dynamique,Evol EPS 0y,0.25,0.20
6,prudent,dynamique,Evol EPS +1y,0.02,0.20
7,prudent,gains,Div %,0.50,0.30
8,prudent,gains,Rendement,0.30,0.30
9,prudent,gains,Potentiel,0.20,0.30


### Définitions des seuils de performance

In [7]:
data_seuils = [
    # --- Profil PRUDENT ---
    ("prudent", "Div %",        0.02, 0.04, 0.01, 0.06),
    ("prudent", "Rendement",   0.02, 0.10, 0.00, 0.20),
    ("prudent", "Volatilité",  0.10, 0.20, 0.00, 0.40),
    ("prudent", "Delta cours moyen", -0.03, 0.03, -0.10, 0.10),
    ("prudent", "Delta MA50", -0.02, 0.02, -0.05, 0.05),
    ("prudent", "Delta ATH",  -0.15, -0.05, -0.25, 0.00),
    ("prudent", "Evol CA 0y",  0.01, 0.05, 0.00, 0.10),
    ("prudent", "Evol CA +1y",  0.02, 0.06, 0.00, 0.12),
    ("prudent", "Evol EPS 0y",  0.02, 0.07, 0.00, 0.15),
    ("prudent", "Evol EPS +1y",  0.03, 0.08, 0.00, 0.15),
    ("prudent", "Potentiel",  0.03, 0.10, 0.00, 0.20),

    # --- Profil ÉQUILIBRÉ ---
    ("équilibré", "Div %",        0.02, 0.06, 0.01, 0.08),
    ("équilibré", "Rendement",   0.05, 0.15, -0.10, 0.30),
    ("équilibré", "Volatilité",  0.10, 0.25, 0.00, 0.50),
    ("équilibré", "Delta cours moyen", -0.05, 0.08, -0.15, 0.15),
    ("équilibré", "Delta MA50", -0.05, 0.05, -0.10, 0.10),
    ("équilibré", "Delta ATH",  -0.20, -0.10, -0.40, 0.10),
    ("équilibré", "Evol CA 0y",  0.03, 0.08, -0.02, 0.15),
    ("équilibré", "Evol CA +1y",  0.04, 0.10, -0.03, 0.18),
    ("équilibré", "Evol EPS 0y",  0.04, 0.12, -0.05, 0.25),
    ("équilibré", "Evol EPS +1y",  0.05, 0.15, -0.05, 0.25),
    ("équilibré", "Potentiel",  0.05, 0.15, -0.05, 0.30),

    # --- Profil DYNAMIQUE ---
    ("dynamique", "Div %",        0.00, 0.03, 0.00, 0.08),
    ("dynamique", "Rendement",   0.10, 0.30, -0.20, 0.60),
    ("dynamique", "Volatilité",  0.20, 0.40, 0.0, 0.80),
    ("dynamique", "Delta cours moyen", -0.10, 0.15, -0.20, 0.25),
    ("dynamique", "Delta MA50", -0.10, 0.10, -0.10, 0.20),
    ("dynamique", "Delta ATH",  -0.25, -0.05, -0.55, 0.25),
    ("dynamique", "Evol CA 0y",  0.05, 0.12, -0.05, 0.20),
    ("dynamique", "Evol CA +1y",  0.08, 0.20, -0.05, 0.30),
    ("dynamique", "Evol EPS 0y",  0.08, 0.25, -0.10, 0.40),
    ("dynamique", "Evol EPS +1y",  0.10, 0.30, -0.10, 0.50),
    ("dynamique", "Potentiel",  0.10, 0.25, -0.10, 0.40),
]

df_seuils = pd.DataFrame(data_seuils, columns=["profil", "indicateur", "obj_min", "obj_max", "cat_min", "cat_max"])

### Définition des intervalles de recommandation

In [8]:
data_intervals = [
    ("strongBuy", 0.00, 0.20, 1),
    ("buy", 0.20, 0.40, 1),
    ("hold", 0.40, 0.60, 0),
    ("sell", 0.60, 0.80, -1),
    ("strongSell", 0.80, 1.00, -1)]
df_intervals = pd.DataFrame(data_intervals, columns=["Recommandation", "score_min", "score_max", "Sens_confiance"])

smiley_map = {
    -2: "😭",
    -1: "😟",
     0: "😐",
     1: "🙂",
     2: "😄"}

def color_reco(val):
    colors = {
        "strongBuy": "#4CAF50",   # vert foncé
        "buy":        "#8BC34A",   # vert clair
        "hold":       "#FFC107",   # jaune
        "sell":       "#FF9800",   # orange
        "strongSell":"#F44336"    # rouge
    }
    return f"background-color: {colors.get(val, 'white')};"

df_intervals

,Recommandation,score_min,score_max,Sens_confiance
0,strongBuy,0.0,0.2,1
1,buy,0.2,0.4,1
2,hold,0.4,0.6,0
3,sell,0.6,0.8,-1
4,strongSell,0.8,1.0,-1


## Calcul des scores de performance

### Calcul des scores de performance individuels

In [9]:
df_scores = df_seuils.copy()
df_scores['valeur'] = df_scores.apply(lambda row: safe_get_value(action_suivie.rendement, row["indicateur"]), axis=1)
df_scores['score'] = df_scores.apply(lambda row: score(row['valeur'], row['obj_min'], row['obj_max'], row['cat_min'], row['cat_max']), axis=1)
df_scores

,profil,indicateur,obj_min,obj_max,cat_min,cat_max,valeur,score
0,prudent,Div %,0.02,0.04,0.01,0.06,0.061397,1.000000
1,prudent,Rendement,0.02,0.10,0.00,0.20,0.468410,1.000000
2,prudent,Volatilité,0.10,0.20,0.00,0.40,0.279542,0.397709
3,prudent,Delta cours moyen,-0.03,0.03,-0.10,0.10,0.338111,1.000000
4,prudent,Delta MA50,-0.02,0.02,-0.05,0.05,0.102627,1.000000
5,prudent,Delta ATH,-0.15,-0.05,-0.25,0.00,-0.001243,0.975133
6,prudent,Evol CA 0y,0.01,0.05,0.00,0.10,0.071200,0.424000
7,prudent,Evol CA +1y,0.02,0.06,0.00,0.12,0.043500,0.000000
8,prudent,Evol EPS 0y,0.02,0.07,0.00,0.15,0.033300,0.000000
9,prudent,Evol EPS +1y,0.03,0.08,0.00,0.15,0.144600,0.922857


### Calcul des scores agrégés

In [10]:
def score_categorie(df_scores, df_poids):
    # Jointure sur profil + indicateur
    merged = df_scores.merge(df_poids, on=["profil", "indicateur"], how="left")
    # Score pondéré par indicateur
    merged["score_indicateur_pondere"] = merged["score"] * merged["poids_indicateur"]
    # Score par catégorie = somme des scores pondérés
    score_cat = merged.groupby(["profil", "categorie"], as_index=False)["score_indicateur_pondere"].sum()
    # Renommage
    score_cat.rename(columns={"score_indicateur_pondere": "score_categorie"}, inplace=True)
    return score_cat

df_score_cat = score_categorie(df_scores, df_poids)
df_score_cat

,profil,categorie,score_categorie
0,dynamique,dynamique,0.205013
1,dynamique,gains,0.448746
2,dynamique,risques,0.205324
3,prudent,dynamique,0.571970
4,prudent,gains,0.800000
5,prudent,risques,0.696368
6,équilibré,dynamique,0.414267
7,équilibré,gains,0.410139
8,équilibré,risques,0.596645


In [11]:
merged = df_score_cat.merge(df_poids[["profil", "categorie", "poids_categorie"]].drop_duplicates(), on=["profil", "categorie"], how="left")
# Score global pondéré
merged["score_global_part"] = merged["score_categorie"] * merged["poids_categorie"]
# Agrégation finale
score_glob = merged.groupby("profil", as_index=False)["score_global_part"].sum()
score_glob.rename(columns={"score_global_part": "score_global"}, inplace=True)
score_glob

,profil,score_global
0,dynamique,0.290366
1,prudent,0.702578
2,équilibré,0.458417


### Interprétation des scores en recommandation

In [12]:
score_glob = score_glob.merge(df_intervals, how='cross')
score_glob

,profil,score_global,Recommandation,score_min,score_max,Sens_confiance
0,dynamique,0.290366,strongBuy,0.0,0.2,1
1,dynamique,0.290366,buy,0.2,0.4,1
2,dynamique,0.290366,hold,0.4,0.6,0
3,dynamique,0.290366,sell,0.6,0.8,-1
4,dynamique,0.290366,strongSell,0.8,1.0,-1
5,prudent,0.702578,strongBuy,0.0,0.2,1
6,prudent,0.702578,buy,0.2,0.4,1
7,prudent,0.702578,hold,0.4,0.6,0
8,prudent,0.702578,sell,0.6,0.8,-1
9,prudent,0.702578,strongSell,0.8,1.0,-1


In [13]:
score_glob = score_glob.loc[(score_glob['score_global'] >= score_glob['score_min']) & (score_glob['score_global'] < score_glob['score_max'])]
score_glob = score_glob.reset_index(drop=True)
score_glob["Confiance"] = 0 + score_glob['Sens_confiance']  # valeur par défaut
score_glob.loc[score_glob["score_global"] <= score_glob["score_min"] + 0.05, "Confiance"] = 1 + score_glob['Sens_confiance']
score_glob.loc[score_glob["score_global"] >= score_glob["score_max"] - 0.05, "Confiance"] = -1 + score_glob['Sens_confiance']
score_glob["Smiley"] = score_glob["Confiance"].map(smiley_map)
score_glob

,profil,score_global,Recommandation,score_min,score_max,Sens_confiance,Confiance,Smiley
0,dynamique,0.290366,buy,0.2,0.4,1,1,🙂
1,prudent,0.702578,sell,0.6,0.8,-1,-1,😟
2,équilibré,0.458417,hold,0.4,0.6,0,0,😐


### Mise en forme des éléments

In [14]:
score_glob.style.map(color_reco, subset=["Recommandation"])

,profil,score_global,Recommandation,score_min,score_max,Sens_confiance,Confiance,Smiley
0,dynamique,0.290366,buy,0.200000,0.400000,1,1,🙂
1,prudent,0.702578,sell,0.600000,0.800000,-1,-1,😟
2,équilibré,0.458417,hold,0.400000,0.600000,0,0,😐


In [18]:
df_cat = df_score_cat[['categorie','score_categorie']].loc[df_score_cat['profil'] == 'prudent']
df_cat

,categorie,score_categorie
3,dynamique,0.571970
4,gains,0.800000
5,risques,0.696368


In [19]:
for t in df_cat['categorie'].unique():
    score = 0.5 - df_cat.loc[df_cat['categorie'] == t, 'score_categorie'].iloc[0]
    color = "green" if score > 0 else "red"
    print(f"Catégorie {t} = {score} - {color}")

Catégorie dynamique = -0.07197047515651034 - red
Catégorie gains = -0.30000000000000004 - red
Catégorie risques = -0.19636792808213877 - red


In [20]:
# Pour chaque profil : index du score minimum
idx_min = df_scores.groupby('profil')['score'].idxmin()

# Pour chaque profil : index du score maximum
idx_max = df_scores.groupby('profil')['score'].idxmax()

# Récupération des lignes correspondantes
df_min = df_scores.loc[idx_min, ['profil', 'indicateur', 'valeur', 'score']]
df_max = df_scores.loc[idx_max, ['profil', 'indicateur', 'valeur', 'score']]

df_min

,profil,indicateur,valeur,score
24,dynamique,Volatilité,0.279542,0.0
7,prudent,Evol CA +1y,0.043500,0.0
17,équilibré,Evol CA 0y,0.071200,0.0


In [21]:
df_extremes = (
    pd.concat([
        df_min.assign(type='score_min'),
        df_max.assign(type='score_max')
    ])
    .sort_values(['profil', 'type'])
    .reset_index(drop=True)
)
df_extremes

,profil,indicateur,valeur,score,type
0,dynamique,Delta cours moyen,0.338111,1.0,score_max
1,dynamique,Volatilité,0.279542,0.0,score_min
2,prudent,Div %,0.061397,1.0,score_max
3,prudent,Evol CA +1y,0.043500,0.0,score_min
4,équilibré,Rendement,0.468410,1.0,score_max
5,équilibré,Evol CA 0y,0.071200,0.0,score_min


In [22]:
df_p = df_extremes[df_extremes['profil'] == 'prudent']
df_p

,profil,indicateur,valeur,score,type
2,prudent,Div %,0.061397,1.0,score_max
3,prudent,Evol CA +1y,0.043500,0.0,score_min


## Modules complémentaires

In [41]:
tickers_paris = pd.read_csv("data/Euronext_Equities.csv", sep=';')
tickers_paris = df[['Name', 'ISIN', 'Symbol','Indice']]
tickers_paris = tickers_paris.copy()
tickers_paris = tickers_paris[tickers_paris['Symbol'].notna()]
tickers_paris = tickers_paris.reset_index(drop=True)
tickers_paris['Ticker'] = tickers_paris['Symbol'].astype(str) + ".PA"
tickers_paris.head()

,Name,ISIN,Symbol,Indice,Ticker
0,2CRSI,FR0013341781,AL2SI,NaN,AL2SI.PA
1,74SOFTWARE,FR0011040500,74SW,NaN,74SW.PA
2,AB SCIENCE,FR0010557264,AB,NaN,AB.PA
3,AB SCIENCE BSA,FR001400ZRT0,ABBS,NaN,ABBS.PA
4,AB SCIENCE BSA 25,FR00140118Z8,ABSBS,NaN,ABSBS.PA


In [42]:
indices = tickers_paris['Indice'].dropna().unique().tolist()
indices.sort()
indices = ["Pas de filtre"] + indices
indices

['Pas de filtre', 'CAC 40', 'MID 60', 'NEXT 20']

In [48]:
filtre_indice = indices[2]
if filtre_indice == 'Pas de filtre':
    tickers_paris_filtres = tickers_paris
else:
    tickers_paris_filtres = tickers_paris.loc[tickers_paris['Indice'] == filtre_indice]
tickers_paris_filtres.reset_index(drop=True)

,Name,ISIN,Symbol,Indice,Ticker
0,ADP,FR0010340141,ADP,MID 60,ADP.PA
1,AIR FRANCE -KLM,FR001400J770,AF,MID 60,AF.PA
2,ALTEN,FR0000071946,ATE,MID 60,ATE.PA
3,ARGAN,FR0010481960,ARG,MID 60,ARG.PA
4,ATOS GROUP,FR001400X2S4,ATO,MID 60,ATO.PA
5,AYVENS,FR0013258662,AYV,MID 60,AYV.PA
6,BIC,FR0000120966,BB,MID 60,BB.PA
7,BOLLORE,FR0000039299,BOL,MID 60,BOL.PA
8,CARMILA,FR0010828137,CARM,MID 60,CARM.PA
9,CLARIANE,FR0010386334,CLARI,MID 60,CLARI.PA
